# DriftSense dataset insights

## tl;dr

This companion notebook reproduces the aggregate analysis used in the HTML report. The outcome is the binary post-session self-report; activity signals are not treated as direct measurements of attention. The supplied files are structurally clean but look synthetic or heavily curated, so findings should not be presented as field-study evidence until provenance is confirmed.

## Context & Methods

The analysis uses one row per labeled browsing session, participant-grouped validation, and participant-clustered bootstrap intervals. It compares the required majority/time, domain, intention-only, activity-only, and combined exploratory baselines.

### Key Assumptions

- `drift_label` is a post-session self-report, not a diagnosis or direct attention measure.
- The CSVs contain final session totals; early 1/3/5-minute evaluation is therefore unavailable.
- The model comparison is exploratory and full-session, not a preregistered result.

## Data

The next cell reruns the standard-library analysis script and refreshes `analysis_summary.json` without creating a row-level combined dataset.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

analysis_dir = Path.cwd()
if analysis_dir.name != 'analysis':
    analysis_dir = Path('driftsense_data_final/analysis')
script_path = analysis_dir / 'analyze_dataset.py'
summary_path = analysis_dir / 'analysis_summary.json'
subprocess.run([sys.executable, str(script_path)], check=True)
summary = json.loads(summary_path.read_text(encoding='utf-8'))
summary['source']

## Results

The following compact views expose the same aggregates used by the report.

In [ ]:
{
    'data_quality': summary['data_quality'],
    'outcome': summary['outcome'],
}

In [ ]:
[
    {
        'intention': item['label'],
        'sessions': item['sessions'],
        'drift_rate': round(item['drift_rate'], 3),
        'participant_bootstrap_ci_95': [round(x, 3) for x in item['ci_95']],
    }
    for item in summary['intentions']
]

In [ ]:
{
    name: {
        metric: round(value, 3) if isinstance(value, float) else value
        for metric, value in result.items()
        if metric in {'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'roc_auc_ci_95_participant_bootstrap'}
    }
    for name, result in summary['exploratory_models'].items()
    if isinstance(result, dict) and 'roc_auc' in result
}

## Takeaways

- Declared intention is the clearest descriptive separator: accidental openings and open-ended browsing have much higher self-reported drift rates than task-directed or planned-break sessions.
- Domain alone is weak, while the same domain can contain very different outcomes under different intentions.
- In exploratory participant-held-out evaluation, intention plus full-session activity ranks sessions better than either family alone, but performance remains modest and is not an early-prediction result.
- Confirm whether the files are synthetic or curated before using any values as empirical study results.